# Modeling Human Activity States Using Hidden Markov Models

## Project Overview

**Background:** In real-world systems—from wearable health monitors to smart home sensors—continuous data streams such as accelerometer and gyroscope signals reveal valuable information about human activity. However, the true activity state (e.g., walking or standing) is often hidden behind noisy measurements.

**Task:** This project collects, analyzes, and models motion data using the Sensor Logger app, then builds a Hidden Markov Model (HMM) to infer human activity states from recorded sensor signals.

**Activities:** Standing, Walking, Jumping, Still (no movement)

**Sensors:** Accelerometer (x, y, z) and Gyroscope (x, y, z)

### Colab Setup: Clone repo to access data

**Run this cell first if using Google Colab** to fetch the data from GitHub.

In [1]:
# Clone the repo to get the data (run this cell first in Colab)
import os
import subprocess

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if not os.path.exists('/content/Hidden_Markov_Models_Formative'):
        subprocess.run(["git", "clone", "https://github.com/1heodora-e/Hidden_Markov_Models_Formative.git"], check=True)
    os.chdir('/content/Hidden_Markov_Models_Formative')
    print("Train contents:", os.listdir('data/Train'))
    print("Test contents:", os.listdir('data/Test'))
else:
    print("Running locally - using existing data folder")

Train contents: ['Victoria', 'Theodora']
Test contents: ['Victoria', 'Theodora']


## 1. Setup and Imports

In [3]:
import os
import re
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
from scipy.fft import fft, fftfreq
from sklearn.preprocessing import StandardScaler, LabelEncoder
!pip install hmmlearn
from hmmlearn import hmm
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 5)
sns.set_style("whitegrid")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.0/166.0 kB 5.2 MB/s eta 0:00:00


## 2. Data Collection & Loading

**Data Collection Notes:**
- Data collected using Sensor Logger app (iOS/Android)
- Sampling rate: ~100 Hz (10 ms interval) - harmonized across devices
- Participants: Theodora, Victoria
- Each recording: 5–10 seconds per activity

In [4]:
# Activity label normalization
ACTIVITY_ALIASES = {
    "phone_still": "still", "stand_still": "still", "stand_still_1": "still", "stand_still_2": "still",
    "test_phone_still": "still", "test_stand_still": "still", "walk": "walking", "test_walk": "walking",
    "jump": "jumping", "test_jump": "jumping", "jumping": "jumping", "walking": "walking",
    "standing": "standing", "still": "still",
}

def normalize_activity(raw: str) -> str:
    n = raw.lower().strip()
    if n.startswith("test_"): n = n[5:]
    n = re.sub(r"[-_\s]+", "_", n)
    return ACTIVITY_ALIASES.get(n, n)

def get_trial_id(path: Path, is_theodora: bool) -> str:
    if is_theodora: return path.parent.name
    stem = path.stem
    stem = re.sub(r"-(Accelerometer|Gyroscope)\s*$", "", stem, flags=re.I)
    stem = re.sub(r"(Accelerometer|Gyroscope)\s*$", "", stem, flags=re.I)
    return stem.strip()

def parse_activity(path: str, is_theodora: bool, participant: str = "") -> str:
    p = Path(path)
    name = p.stem if p.suffix else p.name  # use stem for files (exclude .csv)
    if is_theodora:
        base = re.split(r"-\d{4}-\d{2}-\d{2}_\d{2}-\d{2}-\d{2}", name)[0]
        parts = base.replace("Test_", "").split("_")
        act = "_".join(parts[:-1]) if len(parts) > 1 and parts[-1].isdigit() else "_".join(parts)
    else:
        base = re.sub(r"-(Accelerometer|Gyroscope)\s*$", "", name, flags=re.I)
        base = re.sub(r"(Accelerometer|Gyroscope)\s*$", "", base, flags=re.I)
        # Remove participant prefix (e.g. Victoria_, Victoria )
        if participant and base.lower().replace(" ", "").startswith(participant.lower().replace(" ", "")):
            base = re.sub(r"^" + re.escape(participant) + r"[\s_]*", "", base, flags=re.I)
        parts = re.split(r"[-_\d]+", base)
        act = "_".join(p for p in parts if p).strip("_") or base
    return normalize_activity(act)

In [5]:
def discover_sensor_files(data_dir: str) -> list:
    """Discover accel/gyro files for Victoria (flat) and Theodora (folder) structures."""
    data_path = Path(data_dir)
    records = []
    for participant_dir in data_path.iterdir():
        if not participant_dir.is_dir(): continue
        participant = participant_dir.name
        subdirs = [d for d in participant_dir.iterdir() if d.is_dir()]
        if subdirs and any((d / "Accelerometer.csv").exists() for d in subdirs):
            for rec_dir in subdirs:
                accel = rec_dir / "Accelerometer.csv"
                gyro = rec_dir / "Gyroscope.csv"
                if accel.exists():
                    records.append({
                        "accel_path": str(accel), "gyro_path": str(gyro) if gyro.exists() else None,
                        "activity": parse_activity(str(rec_dir), True, participant),
                        "participant": participant, "trial_id": rec_dir.name, "is_theodora": True,
                    })
        else:
            for accel in participant_dir.glob("*Accelerometer*.csv"):
                gyro_cands = [f for f in participant_dir.glob("*Gyroscope*.csv")
                              if get_trial_id(f, False) == get_trial_id(accel, False)]
                gyro = gyro_cands[0] if gyro_cands else None
                records.append({
                    "accel_path": str(accel), "gyro_path": str(gyro) if gyro else None,
                    "activity": parse_activity(str(accel), False, participant),
                    "participant": participant, "trial_id": get_trial_id(accel, False), "is_theodora": False,
                })
    return records

In [6]:
def load_and_align_trial(record: dict, target_interval_ms: float = 10) -> np.ndarray:
    """Load accel + gyro, align by time, resample to 100Hz. Returns (T, 6) array."""
    accel_df = pd.read_csv(record["accel_path"])
    accel_df.columns = accel_df.columns.str.strip()
    accel = accel_df[["time", "x", "y", "z"]].copy()
    accel.columns = ["time", "acc_x", "acc_y", "acc_z"]

    if record["gyro_path"]:
        gyro_df = pd.read_csv(record["gyro_path"])
        gyro_df.columns = gyro_df.columns.str.strip()
        gyro = gyro_df[["time", "x", "y", "z"]].copy()
        gyro.columns = ["time", "gyro_x", "gyro_y", "gyro_z"]
        tol = np.int64(target_interval_ms * 1.5 * 1e6)
        merged = pd.merge_asof(accel.sort_values("time"), gyro.sort_values("time"),
                               on="time", direction="nearest", tolerance=tol)
        merged = merged.dropna(subset=["gyro_x", "gyro_y", "gyro_z"])
        cols = ["acc_x", "acc_y", "acc_z", "gyro_x", "gyro_y", "gyro_z"]
    else:
        merged = accel
        cols = ["acc_x", "acc_y", "acc_z"]

    merged = merged.sort_values("time")
    t0 = merged["time"].iloc[0]
    merged["t_ms"] = (merged["time"] - t0) / 1e6
    merged = merged.drop_duplicates(subset=["t_ms"], keep="first")
    t_max = merged["t_ms"].max()
    t_target = np.arange(0, t_max + 0.01, target_interval_ms)
    result = np.zeros((len(t_target), len(cols)))
    for i, col in enumerate(cols):
        result[:, i] = np.interp(t_target, merged["t_ms"].values, merged[col].values)
    return result.astype(np.float32)

### Load and Combine All Train CSV Files

All recordings from the Train folder (Victoria + Theodora) are loaded, aligned, and combined into a centralized dataset.

In [7]:
TRAIN_DIR = "data/Train"
records = discover_sensor_files(TRAIN_DIR)
print(f"Found {len(records)} trials in Train folder")
for r in records[:5]:
    print(f"  {r['trial_id']}: {r['activity']} ({r['participant']})")

Found 50 trials in Train folder
  Victoria_still_: still (Victoria)
  Victoria _jumping_4: jumping (Victoria)
  Victoria _Jumping_5: jumping (Victoria)
  Victoria_still-4: still (Victoria)
  Victoria_Jumping_: jumping (Victoria)


In [9]:
# Load all trials and combine into centralized dataset
all_sequences = []
all_activities = []
all_trial_ids = []
all_participants = []

for rec in records:
    try:
        seq = load_and_align_trial(rec)
        if seq.shape[0] < 50:  # skip very short
            continue
        all_sequences.append(seq)
        all_activities.append(rec["activity"])
        all_trial_ids.append(rec["trial_id"])
        all_participants.append(rec["participant"])
    except Exception as e:
        print(f"Skipped {rec['trial_id']}: {e}")

print(f"Loaded {len(all_sequences)} sequences")
print("Activity distribution:", pd.Series(all_activities).value_counts())

# Export combined Train data to single CSV (well-labelled dataset for submission)
combined_rows = []
for seq, act, tid, part in zip(all_sequences, all_activities, all_trial_ids, all_participants):
    has_gyro = seq.shape[1] == 6 # Check if sequence contains gyroscope data
    for t, row in enumerate(seq):
        row_data = {
            "trial_id": tid, "participant": part, "activity": act,
            "acc_x": row[0], "acc_y": row[1], "acc_z": row[2],
        }
        if has_gyro:
            row_data.update({
                "gyro_x": row[3], "gyro_y": row[4], "gyro_z": row[5],
            })
        else:
            row_data.update({
                "gyro_x": np.nan, "gyro_y": np.nan, "gyro_z": np.nan,
            })
        combined_rows.append(row_data)
df_combined = pd.DataFrame(combined_rows)
os.makedirs("data/Train", exist_ok=True)
df_combined.to_csv("data/Train/combined_train_data.csv", index=False)
print(f"\nCombined CSV saved: data/Train/combined_train_data.csv ({len(df_combined)} rows)")

Loaded 50 sequences
Activity distribution: still       13
standing    13
jumping     12
walking     12
Name: count, dtype: int64

Combined CSV saved: data/Train/combined_train_data.csv (42392 rows)
